# 10 End to End - Rollout to Three.js Viewer

## Objective
Validate the source-code pipeline from a fresh runtime: Simulation -> RolloutRecorder -> RolloutData -> export_rollout() -> Healthy_001 -> export_viewer_pose() -> viewer_pose.json -> Three.js viewer.

## Prerequisites
A Google Colab Python runtime with internet access is required so the repository and pinned simulation dependencies can be installed.

## Expected Output
The notebook creates `datasets/healthy/Healthy_001/rollout.json`, `rollout.npz`, `manifest.json`, and `viewer_pose.json`, then launches a local browser viewer.

## Troubleshooting
Every stage uses `assert` statements. If a required file, frame count, quaternion norm, timestamp sequence, or viewer JSON validation fails, execution stops at the failing cell.

## Validation
This notebook verifies non-zero normalized quaternions, exported rollout files, supported NPZ layout, strictly increasing viewer timestamps, and `validate_pose_document()` success.

## Next notebook
This is the final Colab workflow notebook for the rollout-to-viewer path.


## Step 1 - Install package


In [ ]:
from pathlib import Path
import json
import math
import os
import shutil
import subprocess
import sys
import time

REPO_URL = 'https://github.com/TanVi3001/drosophila-pd-flygym.git'
repo = Path.cwd()
if not (repo / 'pyproject.toml').is_file():
    target = Path.cwd() / 'drosophila-pd-flygym'
    if not target.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(target)], check=True)
    os.chdir(target)
else:
    os.chdir(repo)

repo = Path.cwd()
assert (repo / 'pyproject.toml').is_file(), f'pyproject.toml not found in {repo}'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[simulation]'], check=True)
print('Repository ready:', repo)


## Step 2 - Create Fly


In [ ]:
from drosophila_pd.flygym_adapter import FlyGymAdapter, FlyGymConfig

config = FlyGymConfig.from_yaml('configs/v2/flygym/healthy.yaml')
adapter = FlyGymAdapter()
fly = adapter.create_fly(config.fly)
assert fly is not None
assert getattr(fly, 'name', None) == config.fly.name
print('Fly created:', fly.name)


## Step 3 - Create World


In [ ]:
world = adapter.create_world(config.world)
assert world is not None
print('World created:', type(world).__name__)


## Step 4 - Attach Fly


In [ ]:
adapter.attach_fly(
    world,
    fly,
    position=config.world.spawn_position,
    orientation=config.world.spawn_orientation,
    add_ground_contact_sensors=config.world.add_ground_contact_sensors,
)
assert world is not None
print('Fly attached to world')


## Step 5 - Create Simulation


In [ ]:
simulation = adapter.create_simulation(world, config.simulation)
assert simulation is not None
simulation.reset()
assert float(simulation.timestep) > 0
print('Simulation created with timestep:', simulation.timestep)


## Step 6 - Run simulation


In [ ]:
from drosophila_pd.flygym_adapter import FlyGymRuntime, RolloutRecorder

step_count = 25
recorder = RolloutRecorder(
    simulation,
    fly.name,
    fly=fly,
    timestep=float(simulation.timestep),
    simulation_metadata={
        'dataset_id': 'Healthy_001',
        'timestep_s': float(simulation.timestep),
        'source': 'Colab end-to-end notebook',
    },
)
runtime = FlyGymRuntime(simulation, recorder=recorder, max_steps=step_count)
rollout = runtime.run()
assert rollout is recorder.rollout
assert runtime.current_step == step_count
assert rollout.frame_count == step_count + 1
print('Simulation steps:', runtime.current_step)
print('Recorded frames:', rollout.frame_count)


## Step 7 - Validate recorded rollout


In [ ]:
import numpy as np

assert rollout.frame_count > 0
orientations = np.asarray([frame.orientation for frame in rollout.frames], dtype=float)
assert orientations.shape == (rollout.frame_count, 4)
orientation_norms = np.linalg.norm(orientations, axis=1)
assert np.isfinite(orientation_norms).all(), orientation_norms
assert np.all(orientation_norms > 0), orientation_norms
assert np.allclose(orientation_norms, 1.0, atol=1e-6), orientation_norms
timestamps = np.asarray([frame.timestamp_s for frame in rollout.frames], dtype=float)
assert np.isfinite(timestamps).all(), timestamps
print('Quaternion norm min/max:', float(orientation_norms.min()), float(orientation_norms.max()))
print('Timestamp first/last:', float(timestamps[0]), float(timestamps[-1]))


## Step 8 - Export rollout as Healthy_001 dataset


In [ ]:
from drosophila_pd.flygym_adapter import export_rollout

dataset_dir = repo / 'datasets' / 'healthy' / 'Healthy_001'
if dataset_dir.exists():
    shutil.rmtree(dataset_dir)
exported = export_rollout(rollout, dataset_dir)
assert dataset_dir.exists()
for key in ('rollout_json', 'rollout_csv', 'rollout_npz', 'metadata', 'manifest'):
    assert key in exported.files, key
    assert Path(exported.files[key]).exists(), exported.files[key]
arrays = np.load(exported.files['rollout_npz'])
assert 'orientation' in arrays.files
assert 'thorax_quaternions' in arrays.files
assert np.all(np.linalg.norm(arrays['orientation'], axis=1) > 0)
print('Dataset exported:', dataset_dir)
print('NPZ arrays:', sorted(arrays.files))


## Step 9 - Export viewer pose


In [ ]:
from drosophila_pd.viewer_export import export_viewer_pose, validate_pose_document

viewer_pose_path = dataset_dir / 'viewer_pose.json'
pose_result = export_viewer_pose('Healthy_001', viewer_pose_path, search_roots=[repo / 'datasets'])
assert viewer_pose_path.exists()
assert pose_result.validation.overall_pass, pose_result.validation.as_dict()
viewer_pose = json.loads(viewer_pose_path.read_text(encoding='utf-8'))
assert viewer_pose['frame_count'] == rollout.frame_count
assert viewer_pose['frame_count'] > 0
viewer_orientations = np.asarray([frame['orientation'] for frame in viewer_pose['frames']], dtype=float)
viewer_norms = np.linalg.norm(viewer_orientations, axis=1)
assert np.all(viewer_norms > 0), viewer_norms
assert np.allclose(viewer_norms, 1.0, atol=1e-6), viewer_norms
viewer_times = np.asarray([frame['time'] for frame in viewer_pose['frames']], dtype=float)
assert np.all(np.diff(viewer_times) > 0), viewer_times
assert validate_pose_document(viewer_pose).overall_pass
print('Viewer JSON created:', viewer_pose_path)
print('Viewer frame count:', viewer_pose['frame_count'])


## Step 10 - Launch local Three.js viewer


In [ ]:
web_pose = repo / 'web' / 'viewer_pose.json'
shutil.copy2(viewer_pose_path, web_pose)
assert web_pose.exists()
launcher = repo / 'web' / 'colab_viewer.html'
launcher.write_text('''<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Fly Studio Colab Viewer</title>
  <link rel="stylesheet" href="./theme.css">
  <style>body{margin:0;background:#111;color:#eee;font-family:sans-serif}#viewer{width:100vw;height:92vh}.status{padding:8px 12px}</style>
</head>
<body>
  <div class="status" id="status">Loading viewer_pose.json...</div>
  <main id="viewer"></main>
  <script type="module">
    import { Viewer } from './viewer/viewer.js';
    const status = document.getElementById('status');
    const viewer = new Viewer();
    viewer.init(document.getElementById('viewer'));
    const pose = await fetch('./viewer_pose.json').then((response) => {
      if (!response.ok) throw new Error(`viewer_pose.json HTTP ${response.status}`);
      return response.json();
    });
    await viewer.loadPose(pose);
    status.textContent = `Loaded ${pose.frame_count} frames from viewer_pose.json`;
    window.flyStudioViewer = viewer;
  </script>
</body>
</html>
''', encoding='utf-8')
assert launcher.exists()
from urllib.parse import urlparse
ready_file = repo / 'results' / 'colab' / 'viewer_server_url.txt'
server = subprocess.Popen([
    sys.executable,
    'scripts/run_web_demo.py',
    '--host',
    '0.0.0.0',
    '--port',
    '0',
    '--quiet',
    '--ready-file',
    str(ready_file),
])
for _ in range(50):
    if ready_file.exists():
        break
    assert server.poll() is None, 'web server exited early'
    time.sleep(0.1)
assert ready_file.exists(), 'web server did not publish a URL'
viewer_url = ready_file.read_text(encoding='utf-8').strip().replace('/index.html', '/colab_viewer.html')
port = urlparse(viewer_url).port
assert port is not None
assert server.poll() is None, 'web server exited early'
print('Viewer server started:', viewer_url)
try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(port, path='/colab_viewer.html', height=720)
except Exception as exc:
    print('Open this local URL:', viewer_url)
    print('Colab iframe unavailable:', exc)
